# Data Collection and web scraping 

This work aims to automate the collection of real estate data from CoinAfrique using Selenium and BeautifulSoup.

### The url of the document

The url link is the link from the web for scraping data number of rooms, number of bathroom, area, price, location, image_link over 200 pages. The methode used are selenium combined with beautifulSoup.

In [1]:
import os
os.listdir('/kaggle/working')


['__notebook__.ipynb']

In [2]:
url = 'https://ci.coinafrique.com/categorie/appartements?page=13'

### Package required

In [3]:
pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 10.4 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [4]:
# import packages
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup as bs

### Stable Selenium & Chrome Environment Setup for Google Colab
It represents the infrastructure phase of your project, ensuring the web driver is correctly installed before you begin the actual scraping logic.


In [5]:
# Uninstall potentially conflicting chromium packages
!apt-get purge chromium-browser chromium-chromedriver -y
!apt-get autoremove -y

# Install a more recent version of Google Chrome
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install -y google-chrome-stable

# Install unzip utility
!apt-get install -yqq unzip

# Get the installed Chrome version
CHROME_VERSION = !google-chrome --product-version
CHROME_VERSION = CHROME_VERSION[0].strip()

# Download the compatible Chromedriver using the exact Chrome version for consistency
# The URL uses the 'chrome-for-testing' endpoint for reliable downloads.
!wget --no-verbose -O /tmp/chromedriver_linux64.zip "https://edgedl.me.gvt1.com/edgedl/chrome/chrome-for-testing/{CHROME_VERSION}/linux64/chromedriver-linux64.zip"

# Unzip and move chromedriver to a standard path
!unzip -o /tmp/chromedriver_linux64.zip -d /tmp/
!chmod +x /tmp/chromedriver-linux64/chromedriver
!mv /tmp/chromedriver-linux64/chromedriver /usr/bin/chromedriver

# import necessary Service for updated Selenium
from selenium.webdriver.chrome.service import Service

# instantiate a Chrome options object
options = webdriver.ChromeOptions()
# set the options to use Chrome in headless mode (used for running the script in the background)
options.add_argument('--headless')
# Add no-sandbox argument for Colab environment
options.add_argument('--no-sandbox')
# Add disable-dev-shm-usage argument to prevent issues in Colab
options.add_argument('--disable-dev-shm-usage')

# Create a Service object with the executable path
service = Service(executable_path='/usr/bin/chromedriver')

# initialize an instance of the Chrome driver (browser) in headless mode, passing the service object
driver = webdriver.Chrome(service=service, options=options)




Package 'chromium-browser' is not installed, so not removed
Package 'chromium-chromedriver' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.



0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
OK
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,213 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packa

### Data Extraction and Content Parsing Phase
This code navigates the automated browser to a specific search page, captures the live HTML structure, and uses BeautifulSoup to isolate and count every individual apartment listing via their CSS classes, effectively validating that the data is accessible and ready for extraction.

In [6]:

url = 'https://ci.coinafrique.com/categorie/appartements?page=13'
# get the url 
driver.get(url)
# get the content 
soup = bs(driver.page_source, 'html.parser')
# find containers 
containers = soup.find_all('div', 'col s6 m4 l3')
print(len(containers))

84


In [7]:
container = containers[0]


### Deep-Level Navigation
This code constructs the full URL for a specific listing, navigates the browser to its dedicated detail page, and captures the source code to prepare for scraping granular data like square footage or bathroom counts, that isn't visible on the main results page.

In [8]:
# url sous page
url_sous_page = 'https://ci.coinafrique.com' + container.find('a')['href']
# get the url
driver.get(url_sous_page)
res_sous_page = driver.page_source
# find sous-page
soup_page = bs(res_sous_page, 'html.parser')

### Specific Detail Extraction
This code targets the property's technical characteristics by locating the details container, isolating individual list items, and parsing specific spans to retrieve the number of bathrooms, rooms, and the total area while automatically cleaning the text to remove units like "m2."

In [9]:
infos = soup_page.find('div', 'details-characteristics')
info = infos.find_all('li')

# Construire le dict en one-liner
characteristics = {li.find_all('span')[0].text.strip(): li.find_all('span')[1].text.strip() for li in info}

# Extraire comme avant
num_of_rooms     = characteristics.get('Nbre de pièces', 'N/A')
num_of_bathrooms = characteristics.get('Nbre de salles de bain', 'N/A')
area             = characteristics.get('Superficie', 'N/A').replace('m2', '').strip()

### Deep Navigation & Data Targeting
This code reconstructs the complete URL of an individual listing to access its private detail page, captures the source code, and specifically targets the "details-characteristics" section to isolate the raw list of technical specifications for further extraction.

In [10]:
# Retrieve the URL of the subpage: search for the link in the container,
# extract its attribute, and thus access the detailed page.
url_sous_page = 'https://ci.coinafrique.com' + container.find('a')['href']
driver.get(url_sous_page)
res_sous_page = driver.page_source
soup_page = bs(res_sous_page, 'html.parser')

#Extract a specific table from the page.
infos = soup_page.find('div', 'details-characteristics')
info = infos.find_all('li')

## Full-Scale Multi-Page Web Scraper
This code implements a robust automation loop that iterates through 50 pages of listings, systematically navigating into each individual apartment's sub page to extract high-precision data—including prices, locations, room counts, and surface areas—before consolidating all the findings into a structured Pandas DataFrame for final analysis.


In [11]:
df = pd.DataFrame()
data = []
for index in range(1, 200):
    url = f'https://ci.coinafrique.com/categorie/appartements?page={index}'
    # get the url
    driver.get(url)
    # get the content
    soup = bs(driver.page_source, 'html.parser')
    # find containers
    containers = soup.find_all('div', 'col s6 m4 l3')
    for container in containers:

        # Retrieve the URL of the subpage: search for the link in the container,
        # extract its attribute, and thus access the detailed page.
        url_sous_page = 'https://ci.coinafrique.com' + container.find('a')['href']
        driver.get(url_sous_page)
        res_sous_page = driver.page_source
        soup_page = bs(res_sous_page, 'html.parser')

        #Extract a specific table from the sous-page.
        infos = soup_page.find('div', 'details-characteristics')

        try:

            info = infos.find_all('li')
            # scrape the price, price, adress
            details = container.find('p', class_ = 'ad__card-description').text
            price = container.find('p', class_ ='ad__card-price').text.replace("CFA", "")
            adress = container.find('p', class_ = 'ad__card-location').span.text

            # scraper the numbers of bathrooms, rooms and area
            num_of_bathrooms = info[0].find_all('span')[1].text
            num_of_rooms = info[1].find_all('span')[1].text
            area = "".join(info[2].find_all('span')[1].text.replace('m2', ''))

            #image link
            image_links = 'https://ci.coinafrique.com' + container.find('a')['href']

            #
            dic = {
                'details':details,
                'price':price,
                'adress':adress,
                'num_of_bathrooms':num_of_bathrooms,
                'num_of_rooms':num_of_rooms,
                'area':area,
                'image_links':image_links
            }

            data.append(dic)

        except:
            pass
# All extracted data
data_scrape = pd.DataFrame(data)
df = pd.concat([df, data_scrape], axis = 0).reset_index(drop = True)

In [12]:
df.head()

,details,price,adress,num_of_bathrooms,num_of_rooms,area,image_links
0,Location appartement 3 pièces - Cocody riviera...,350 000,"Cocody, Abidjan, Côte d'Ivoire",3,3,120,https://ci.coinafrique.com/annonce/appartement...
1,Vente appartement 4 pièces à Cocody Angre Chât...,40 000 000,"Cocody, Abidjan, Côte d'Ivoire",4,4,150,https://ci.coinafrique.com/annonce/appartement...
2,Vente appartement 4 pièces à Cocody Angre 8em...,150 000 000,"Cocody, Abidjan, Côte d'Ivoire",4,4,155,https://ci.coinafrique.com/annonce/appartement...
3,Location appartement 3 pièces - Cocody rivier...,350 000,"Cocody, Abidjan, Côte d'Ivoire",3,3,120,https://ci.coinafrique.com/annonce/appartement...
4,Location appartement - Cocody,280 000 000,"Cocody, Abidjan, Côte d'Ivoire",4,4,161,https://ci.coinafrique.com/annonce/appartement...


In [13]:
df.shape

(183, 7)

## Data Cleanning
For this part, we try to check the missing value, duplicated, check the type of variable. We trying to fix it for making data useful.

In [14]:
# Checking columns
df.columns

Index(['details', 'price', 'adress', 'num_of_bathrooms', 'num_of_rooms',
       'area', 'image_links'],
      dtype='object')

In [15]:
# Checking the missing values
df.isnull().sum()

details             0
price               0
adress              0
num_of_bathrooms    0
num_of_rooms        0
area                0
image_links         0
dtype: int64

In [16]:
# Checking the duplicates
df.duplicated().sum()

np.int64(1)

In [17]:
# Drop the duplicates
df.drop_duplicates(inplace = True)

In [18]:
#checking the types
df.dtypes

details             object
price               object
adress              object
num_of_bathrooms    object
num_of_rooms        object
area                object
image_links         object
dtype: object

In [19]:
# Change the types
df['num_of_bathrooms'] = df['num_of_bathrooms'].astype(int)
df['num_of_rooms'] = df['num_of_rooms'].astype(int)
df['area'] = df['area'].astype(float)

In [20]:
df['price'] = df['price'].str.strip()

In [21]:
df['price'].unique()

array(['350 000', '40 000 000', '150 000 000', '280 000 000', '700 000',
       '1 000 000', '400 000', '600 000', '140 000 000', '95 000 000',
       '1 500 000', '250 000', '2 000 000', '550 000', '850 000',
       '2 500 000', '1 800 000', '85 000 000', '450 000', '900 000',
       '530 000', '2 200 000', '1 600 000', '165 000 000', '30 000 000',
       '500 000', '38 000 000', '90 000 000', '60 000 000', '800 000',
       '650 000', '950 000', '300 000', '1 900 000', '200 000', '835 000',
       '750 000', '1 100 000', '205 000 000', '180 000', '110 000',
       '100 000 000', '450 000 000', '1 300 000', '410 000',
       '230 000 000', '1 150 000', '150 000', '270 000', '1 400 000',
       '1 200 000', '230 000', '55 000 000', '70 000', '49 000 000',
       '75 000 000', '100 000', '130 000 000', '37 000 000', '380 000'],
      dtype=object)

In [22]:
df = df[df['price'] != 'Prix sur demande']

In [23]:
df['price'] = df['price'].str.replace(' ', '')

In [24]:
df['price'] = df['price'].astype(int)

In [25]:
# Checking the types
df.dtypes

details              object
price                 int64
adress               object
num_of_bathrooms      int64
num_of_rooms          int64
area                float64
image_links          object
dtype: object

In [26]:
df.head()

,details,price,adress,num_of_bathrooms,num_of_rooms,area,image_links
0,Location appartement 3 pièces - Cocody riviera...,350000,"Cocody, Abidjan, Côte d'Ivoire",3,3,120.0,https://ci.coinafrique.com/annonce/appartement...
1,Vente appartement 4 pièces à Cocody Angre Chât...,40000000,"Cocody, Abidjan, Côte d'Ivoire",4,4,150.0,https://ci.coinafrique.com/annonce/appartement...
2,Vente appartement 4 pièces à Cocody Angre 8em...,150000000,"Cocody, Abidjan, Côte d'Ivoire",4,4,155.0,https://ci.coinafrique.com/annonce/appartement...
3,Location appartement 3 pièces - Cocody rivier...,350000,"Cocody, Abidjan, Côte d'Ivoire",3,3,120.0,https://ci.coinafrique.com/annonce/appartement...
4,Location appartement - Cocody,280000000,"Cocody, Abidjan, Côte d'Ivoire",4,4,161.0,https://ci.coinafrique.com/annonce/appartement...


## Data recording

In [27]:
df.to_csv('data_clean.csv', index = False)